# Synthetic Math Page Generation Examples

This notebook shows how to control page type, formula density, scan/photo distortions, and MathWriting split selection. It uses the tiny fixture dataset by default so it can run anywhere. For real generation, point `MATHWRITING_ROOT` to the MathWriting 2024 dataset root.

In [ ]:
from pathlib import Path
import json
import tempfile

from IPython.display import display
from PIL import Image

from mathglyph_pages import MathPageConfig, generate_pages

cwd = Path.cwd()
if (cwd / "tests" / "fixtures" / "tiny_mathwriting").exists():
    REPO_ROOT = cwd
elif (cwd.parent / "tests" / "fixtures" / "tiny_mathwriting").exists():
    REPO_ROOT = cwd.parent
else:
    raise RuntimeError("Run this notebook from the repo root or notebooks directory")

MATHWRITING_ROOT = REPO_ROOT / "tests" / "fixtures" / "tiny_mathwriting"
WORKDIR_HANDLE = tempfile.TemporaryDirectory()
WORKDIR = Path(WORKDIR_HANDLE.name)
print("MathWriting root:", MATHWRITING_ROOT)
print("Temporary output dir:", WORKDIR)

In [ ]:
def run_example(name: str, **kwargs):
    config = MathPageConfig(
        mathwriting_root=MATHWRITING_ROOT,
        out_dir=WORKDIR / name,
        splits=("train", "synthetic"),
        max_scan_per_split=None,
        **kwargs,
    )
    result = generate_pages(config)
    summary = json.loads(result.summary_path.read_text(encoding="utf-8"))
    print(name, "pages=", summary["num_pages"], "regions=", summary["num_regions"])
    print("profiles:", summary["page_profiles"])
    print("styles:", summary["visual_styles"])
    if result.contact_sheet_path:
        display(Image.open(result.contact_sheet_path))
    return result, summary

## Clean sparse pages

Use `profile="formula_sparse"` for fewer formulas and `visual_style="clean"` for minimal augmentation.

In [ ]:
clean_sparse, clean_sparse_summary = run_example(
    "clean_sparse",
    num_pages=2,
    seed=101,
    profile="formula_sparse",
    visual_style="clean",
    formulas_per_page_min=4,
    formulas_per_page_max=5,
)

## Dense pages

Density is controlled by the page profile and the explicit formula-count bounds.

In [ ]:
dense, dense_summary = run_example(
    "dense",
    num_pages=2,
    seed=202,
    profile="formula_dense",
    visual_style="noisy_scan",
    formulas_per_page_min=12,
    formulas_per_page_max=14,
    formula_target_height_min=62,
    formula_target_height_max=88,
)

## Distortion controls

Choose a visual preset or override individual distortion fields directly.

In [ ]:
distorted, distorted_summary = run_example(
    "distorted",
    num_pages=2,
    seed=303,
    profile="formula_scattered",
    visual_style="clean",
    rotation_max_degrees=3.0,
    perspective_strength=0.05,
    yellow_strength=0.18,
    noise_strength=11,
    blur_radius=0.15,
    edge_shadow=True,
)

## Tables and matrix-heavy pages

`formula_matrix_table` requests matrix/array labels from MathWriting for part of the page.

In [ ]:
matrix_table, matrix_table_summary = run_example(
    "matrix_table",
    num_pages=1,
    seed=404,
    profile="formula_matrix_table",
    visual_style="aged_scan",
    formulas_per_page_min=6,
    formulas_per_page_max=8,
)

## Inspect metadata

Every formula region records which MathWriting split/file/sample drove it.

In [ ]:
rows = [json.loads(line) for line in matrix_table.metadata_path.read_text(encoding="utf-8").splitlines()]
formula_regions = [region for region in rows[0]["regions"] if region["type"] == "formula"]
formula_regions[:3]